In [0]:
src = dp["src_catalog"]

# Mapeo cuenta -> cliente del mes, SIN filtro de CODAPP (patrón de la fuente: CODMES derivado de FECDIA)
ctas = (spark.table(f"{src}.bcp_udv_int_vu.h_cuentafinanciera")
        .withColumn("CODMES", F.date_format(F.col("FECDIA"), "yyyyMM").cast("int"))
        .filter(F.col("CODMES") == 202512)
        .select(F.trim(F.col("CODCLAVECTA")).alias("codclavecta"),
                F.trim(F.col("CODCLAVEPARTYCLI")).alias("codclavepartycli"))
        .dropDuplicates(["codclavecta"]))

tc = (spark.table(f"{src}.bcp_udv_int_vu.h_saldocuentatarjetacredito")
      .withColumn("CODMES", F.date_format(F.col("FECSALDO"), "yyyyMM").cast("int"))
      .filter(F.col("CODMES") == 202512)
      .select(F.trim(F.col("CODCLAVECTA")).alias("codclavecta"),
              F.col("CTDDIAMOROSO").cast("int").alias("dias")))

cp = (spark.table(f"{src}.bcp_udv_int_vu.h_saldocuentacreditopersonal")
      .withColumn("CODMES", F.date_format(F.col("FECSALDO"), "yyyyMM").cast("int"))
      .filter(F.col("CODMES") == 202512)
      .select(F.trim(F.col("CODCLAVECTA")).alias("codclavecta"),
              F.col("CTDDIAVCDA").cast("int").alias("dias")))

raw = (tc.unionByName(cp).join(ctas, "codclavecta")
       .groupBy("codclavepartycli").agg(F.max("dias").alias("max_raw")))

dif.join(raw, "codclavepartycli", "left").select(
    F.count("*").alias("n_dif"),
    F.sum(F.when(F.col("max_raw")==F.col("arnold"),1).otherwise(0)).alias("match_raw"),
    F.sum(F.when(F.col("max_raw")> F.col("arnold"),1).otherwise(0)).alias("raw_mayor"),
    F.sum(F.when(F.col("max_raw")< F.col("arnold"),1).otherwise(0)).alias("raw_menor"),
    F.sum(F.when(F.col("max_raw").isNull(),1).otherwise(0)).alias("sin_fila"),
).show()